## Implement Databricks Subagents with LangChain

### Installing Utilities and Libraries

In [ ]:
%pip install -U \
    databricks-langchain==0.20.0 \
    langgraph==1.2.11 \
    databricks-mcp==0.9.2 \
    "mcp>=1.9"

### Restart the Python Environment

In [ ]:
dbutils.library.restartPython()

### Setup MLflow Tracing

In [ ]:
import mlflow
import os

# Enable auto-tracing for OpenAI
mlflow.openai.autolog()

# Set up MLflow tracking to Databricks
mlflow.set_tracking_uri("databricks")
mlflow.set_experiment("/Shared/Subagents-trace")

### Instantiate the ChatDatabricks Class

In [ ]:
import json
from databricks_langchain import ChatDatabricks

model = ChatDatabricks(
    endpoint="databricks-gpt-oss-120b",
    temperature=0.1,
    max_tokens=20000,
)

### Load the Web Search MCP Tool

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks_langchain import DatabricksMCPServer, DatabricksMultiServerMCPClient

w = WorkspaceClient()

workspace_client = WorkspaceClient()

web_search_mcp_url = "https://adb-7405615206942123.3.azuredatabricks.net/ai-gateway/mcp-services/db_context_engineer_workspace.default.web-search-mcp-server"

mcp_client = DatabricksMultiServerMCPClient([
    DatabricksMCPServer(
        name="web_search",
        url=web_search_mcp_url,
        workspace_client=workspace_client,
    )
])

mcp_tools = await mcp_client.get_tools()

for tool in mcp_tools:
    print(tool.name)

### Create the Researcher Subagent

In [ ]:
from langchain.agents import create_agent

research_agent = create_agent(
    model=model,
    tools=mcp_tools,
    system_prompt="""
You are a senior market research analyst.

Use web search whenever current information,
competitor research, or market trends are required.

Always cite your findings.
"""
)

### Create the Content Writer Subagent

In [ ]:
content_writer = create_agent(
    model=model,
    tools=[],
    system_prompt="""
You are an expert Marketing Content Writer.

Create:

- LinkedIn posts
- Product launch announcements
- Marketing copy
- Promotional content

Write in a professional and engaging tone.
"""
)

### Wrap the Subagents as tools

In [ ]:
from langchain.tools import tool
from mlflow.entities import SpanType

@tool
@mlflow.trace(name="research", span_type=SpanType.TOOL)
async def research(query: str) -> str:
    """
    Research a topic and return findings.
    """

    print("\n" + "=" * 60)
    print("Executing Research Agent")
    print("=" * 60)
    print(query)
    print()

    result = await research_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": query
                }
            ]
        }
    )

    answer = result["messages"][-1].content

    print("\nResearch Agent Finished.\n")

    return answer


@tool
@mlflow.trace(name="write_marketing_copy", span_type=SpanType.TOOL)
async def write_marketing_copy(prompt: str) -> str:
    """
    Create marketing content.
    """

    print("\n" + "=" * 60)
    print("Executing Content Writer Agent")
    print("=" * 60)
    print(prompt)
    print()

    result = await content_writer.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        }
    )

    answer = result["messages"][-1].content

    print("\nContent Writer Finished.\n")

    return answer

### Create the Supervisor Agent

In [ ]:
supervisor = create_agent(
    model=model,

    tools=[
        research,
        write_marketing_copy
    ],

    system_prompt="""
You are the Marketing Supervisor.

Delegate work to the appropriate specialist.

Use:

- research()
    For customer analysis, competitors, personas, positioning.

- write_marketing_copy()
    For LinkedIn posts, launch announcements and promotional content.

Combine the results into one final response.
"""
)

### Invoke the Supervisor Agent

In [ ]:
@mlflow.trace
async def run_workflow(user_query: str):
    response = await supervisor.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": """
    We are launching a new AI-powered fitness smartwatch called FitSense AI.

    First identify:

    - Target audience
    - Customer pain points
    - Current fitness wearable trends

    Then create a professional LinkedIn launch announcement.
    """
                }
            ]
        }
    )

    print("\n" + "=" * 60)
    print("FINAL RESPONSE")
    print("=" * 60)
    return response["messages"][-1].text

In [ ]:
import nest_asyncio
import asyncio

nest_asyncio.apply()

def predict_fn(query: str) -> str:
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_workflow(query))

In [ ]:
user_query = """  We are launching a new AI-powered fitness smartwatch called FitSense AI.

    First identify:

    - Target audience
    - Customer pain points
    - Current fitness wearable trends

    Then create a professional LinkedIn launch announcement. """

print(predict_fn(user_query))